# 4. Investigate event-level errors, then decide whether to open final assessment
No point adjustment. One incident can match at most one fault. Duplicate incidents
remain workload. Detection delay uses the observable-onset proxy, with physical
onset retained in truth. Warning opportunity requires at least three consecutive
observed Rx readings after observable onset and at least 30 minutes before impact.
This denominator does not depend on detector scores or the chosen debounce policy.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation
from optical_anomaly.workflow import run_split

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
split = run_split(settings)
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    split.train_end,
    split.calibration_end,
    split.validation_end,
    split.test_end,
]
REPORT = RUN / "eda"
REPORT.mkdir(exist_ok=True)
print("Run:", RUN.resolve())

`evaluation` in the configuration fixes the opportunity and warning-time policy.
`pre_impact_recall` credits any pre-impact warning; `minimum_lead_recall` requires
the configured lead time (30 minutes by default). The latter selects policies.
Read both alongside recall over all impacted faults, misses and score coverage.
A missing or ineligible warning opportunity must not disappear from the report.

In [ ]:
outcomes = pd.read_csv(RUN / "validation_faults.csv")
display(
    outcomes.groupby("fault_type").agg(
        faults=("detected", "size"),
        detected=("detected", "sum"),
        opportunities=("opportunity", "sum"),
        pre_impact=("pre_impact", "sum"),
        minimum_lead=("minimum_lead", "sum"),
        median_lead_minutes=("lead_minutes", "median"),
    )
)
display(outcomes.loc[~outcomes.detected])
display(pd.read_csv(RUN / "validation_incidents.csv").head(12))

comparison = pd.read_csv(RUN / "validation_comparison.csv")
display(comparison.groupby(["feature_set", "detector"]).agg(
    feasible=("meets_workload_budget", "sum"),
    nuisance_min=("nuisance_per_1000_entity_days", "min"),
    coverage=("score_coverage", "first"),
    duplicate_min=("duplicate_incidents", "min"),
    gap_closures_min=("telemetry_gap_closures", "min"),
))
incidents = pd.read_csv(RUN / "validation_incidents.csv")
display(incidents.reason.value_counts())
display(outcomes.groupby("opportunity_reference").agg(
    faults=("detected", "size"),
    detected=("detected", "sum"),
    opportunities=("opportunity", "sum"),
))

In [ ]:
OPEN_FINAL_TEST = False
if OPEN_FINAL_TEST:
    display(pd.Series(final_evaluation(RUN)))
else:
    print("Final performance assessment remains unopened.")

Final assessment reuses the frozen model and policy. Code/data/model checksums
prevent accidental changes; a one-opening marker prevents accidental repeat use.
These are local safeguards, not a security boundary. Once inspected, final data
cannot serve as an untouched test for further tuning. Faults crossing split
boundaries are reported separately, not silently treated as misses or successes.

The `delay_reference` column in `validation_faults.csv` identifies the time origin.
Variance-shift delay
is measured from physical onset and reported separately from observable-onset
delay. Variance-shift warning opportunities use physical onset and are reported
separately from observable-onset opportunities; physical onset does not establish
that the change was detectable at that instant. Persistent low power remains an impact proxy,
not a measured customer outage.